In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import sem

# Load both datasets
# Ensure these files are in your working directory
try:
    df_bin = pd.read_csv("binary_data_all_V3.csv")
    df_mc = pd.read_csv("multi_class_data_all_V2.csv")
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure the CSV files are present.")
    # Exiting or handling as needed
    exit()

# Define specific thresholds for each metric and dataset combination
thresholds = {
    'bin_ERT_L1_ECE': 0.02,
    'mc_ERT_generalized_norm_score_2': 0.02,
}

methods_config = {
    "tabICLv2": {"label": "TabICLv2", "include": True},
    "TabPFN": {"label": "RealTabPFN-2.5", "include": True},
    "BetterCatBoost": {"label": "CatBoost", "include": True},
    "CheapBetterLGBMClassifier": {"label": "LightGBM", "include": True},
    "RF": {"label": "RandomForest", "include": False},
    "XT": {"label": "ExtraTrees", "include": False},
    "PartitionWise": {"label": "PartitionWise", "include": True},
    "NadarayaWatson": {"label": "Nadaraya-Watson", "include": True},
    "InitLogitLGBMClassifier": {"label": "WS LGBM", "include": True},
    "InitLogitCatboostClassifier": {"label": "WS CatBoost", "include": True},
    "InitLogitXGBClassifier": {"label": "WS XGB", "include": False},
    "IsotonicPredictor": {"label": "Isotonic", "include": True},
    "SMSPredictor": {"label": "SMS", "include": False},
    "TSPredictor": {"label": "TS", "include": True},
}

valid_methods = [k for k, v in methods_config.items() if v.get('include', False)]
group_cols = ['dataset', 'config']
clip = True

def get_metrics(df, metrics_list, primary_metric, prefix=""):
    """
    Computes mean percentages relative to the max for a list of metrics.
    Rows are filtered based ONLY on the threshold of the 'primary_metric'.
    """
    results = {m: {} for m in valid_methods}
    
    # Identify all necessary columns first
    all_needed_cols = []
    for m in metrics_list:
        all_needed_cols.extend([f"{m}_{meth}" for meth in valid_methods if f"{m}_{meth}" in df.columns])
    
    # Group and aggregate
    df_avg = df.groupby(group_cols)[all_needed_cols].mean().reset_index()
    if clip:
        df_avg[all_needed_cols] = df_avg[all_needed_cols].clip(lower=0)
        
    # Determine the 'best' value for the PRIMARY metric for thresholding
    primary_cols = [f"{primary_metric}_{m}" for m in valid_methods if f"{primary_metric}_{m}" in df_avg.columns]
    df_avg['primary_best_val'] = df_avg[primary_cols].max(axis=1)
    
    # Filter: Keep ONLY rows where the primary metric exceeds its threshold
    primary_threshold = thresholds.get(f"{prefix}{primary_metric}", 0.0)
    df_filtered = df_avg[df_avg['primary_best_val'] > primary_threshold].copy()
    
    # Now process each metric using the filtered indices
    for metric in metrics_list:
        metric_cols = [f"{metric}_{m}" for m in valid_methods if f"{metric}_{m}" in df_filtered.columns]
        
        # Calculate max value for the current metric to compute relative percentages
        df_filtered['current_metric_max'] = df_filtered[metric_cols].max(axis=1)
        
        for m in valid_methods:
            col_name = f"{metric}_{m}"
            if col_name in df_filtered.columns:
                # Calculate percentages relative to the max of THIS specific metric
                percentages = (df_filtered[col_name] / df_filtered['current_metric_max']) * 100
                mean_val = percentages.mean()
                se_val = percentages.sem() 
                
                results[m][f"{prefix}{metric}"] = f"${mean_val:.1f}_{{{se_val:.1f}}}$"
                results[m][f"{prefix}{metric}_raw"] = mean_val 
            else:
                results[m][f"{prefix}{metric}"] = "-"
                results[m][f"{prefix}{metric}_raw"] = float('-inf') 
                
    return results

# Define lists and specify the primary metric for each dataset type
metrics_to_extract_bin = ['ERT_L1_ECE', 'ERT_brier_score']
res_bin = get_metrics(df_bin, metrics_to_extract_bin, primary_metric='ERT_L1_ECE', prefix="bin_")

metrics_to_extract_mc = ['ERT_generalized_norm_score_2', 'ERT_brier_score']
res_mc = get_metrics(df_mc, metrics_to_extract_mc, primary_metric='ERT_generalized_norm_score_2', prefix="mc_")

# Rest of the script remains similar, combining results and generating LaTeX
df_all = pd.concat([df_bin, df_mc], ignore_index=True)
time_metric = 'time'
time_cols = [f"{time_metric}_{m}" for m in valid_methods if f"{time_metric}_{m}" in df_all.columns]
df_time = df_all[group_cols + time_cols + ['n_samples']].copy()

for col in time_cols:
    df_time[col] = df_time[col] / (df_time['n_samples'] / 1000)

df_time_avg = df_time.groupby(group_cols)[time_cols].mean().reset_index()

res_time = {m: {} for m in valid_methods}
for m in valid_methods:
    col_name = f"{time_metric}_{m}"
    if col_name in df_time_avg.columns:
        mean_time = df_time_avg[col_name].mean()
        se_time = df_time_avg[col_name].sem()
        res_time[m][time_metric] = f"${mean_time:.1f}_{{{se_time:.1f}}}$"
    else:
        res_time[m][time_metric] = "-"

results_data = {m: {} for m in valid_methods}
for m in valid_methods:
    results_data[m].update(res_bin[m])
    results_data[m].update(res_mc[m])
    results_data[m].update(res_time[m])

for m in valid_methods:
    raw_scores = [
        results_data[m].get('bin_ERT_L1_ECE_raw', np.nan),
        results_data[m].get('bin_ERT_brier_score_raw', np.nan),
        results_data[m].get('mc_ERT_generalized_norm_score_2_raw', np.nan),
        results_data[m].get('mc_ERT_brier_score_raw', np.nan)
    ]
    results_data[m]['overall_avg'] = np.nanmean(raw_scores)

sorted_methods = sorted(
    valid_methods, 
    key=lambda m: results_data[m].get('overall_avg', float('-inf')), 
    reverse=True
)

latex_lines = []
latex_lines.append(r"\begin{table}[h!]")
latex_lines.append(r"\centering")
latex_lines.append(r"\resizebox{\linewidth}{!}{")
latex_lines.append(r"\begin{tabular}{l cc cc c}")
latex_lines.append(r"\toprule")
latex_lines.append(r"& \multicolumn{2}{c}{Avg. \% of max CE} & \multicolumn{2}{c}{Avg. \% of max CE} & \\")
latex_lines.append(r"\cmidrule(lr){2-3} \cmidrule(lr){4-5}")
latex_lines.append(r"Classifier & $\mathrm{CE}_{|\cdot|}$ & $\mathrm{CE}_{(\cdot)^2}$ & $\mathrm{CE}_{\|\cdot\|_2}$ & $\mathrm{CE}_{\|\cdot\|_2^2}$ & \makecell{Avg. time per\\1K samples [s]} \\")
latex_lines.append(r"\midrule")

for m in sorted_methods:
    label = methods_config[m]["label"]
    bin_l1 = results_data[m].get('bin_ERT_L1_ECE', '-')
    bin_brier = results_data[m].get('bin_ERT_brier_score', '-')
    mc_l2 = results_data[m].get('mc_ERT_generalized_norm_score_2', '-')
    mc_brier = results_data[m].get('mc_ERT_brier_score', '-')
    time_str = results_data[m].get('time', '-')
    
    row_str = f"{label} & {bin_l1} & {bin_brier} & {mc_l2} & {mc_brier} & {time_str} \\\\"
    latex_lines.append(row_str)

latex_lines.append(r"\bottomrule")
latex_lines.append(r"\end{tabular}")
latex_lines.append(r"}")
latex_lines.append(r"\caption{\textbf{CE recovered by different methods}, relative to the highest value among all methods for binary (left) and multiclass (right). Experiments are repeated 10 times, and the index number is the standard error across all datasets. Methods are ranked based on the highest average over the four columns.}")
latex_lines.append(r"\label{table:percentage:improvement}")
latex_lines.append(r"\end{table}")

latex_table = "\n".join(latex_lines)
print(latex_table)



\begin{table}[h!]
\centering
\resizebox{\linewidth}{!}{
\begin{tabular}{l cc cc c}
\toprule
& \multicolumn{2}{c}{Avg. \% of max CE} & \multicolumn{2}{c}{Avg. \% of max CE} & \\
\cmidrule(lr){2-3} \cmidrule(lr){4-5}
Classifier & $\mathrm{CE}_{|\cdot|}$ & $\mathrm{CE}_{(\cdot)^2}$ & $\mathrm{CE}_{\|\cdot\|_2}$ & $\mathrm{CE}_{\|\cdot\|_2^2}$ & \makecell{Avg. time per\\1K samples [s]} \\
\midrule
TabICLv2 & $71.6_{1.7}$ & $57.5_{2.3}$ & $72.7_{3.3}$ & $66.7_{3.8}$ & $3.7_{0.1}$ \\
RealTabPFN-2.5 & $70.5_{1.7}$ & $52.8_{2.4}$ & $72.5_{3.1}$ & $59.5_{3.9}$ & $4.7_{0.1}$ \\
InitLogit CatBoost & $72.9_{1.7}$ & $59.4_{2.2}$ & $61.9_{3.3}$ & $60.5_{3.9}$ & $4.2_{0.1}$ \\
InitLogit LGBM & $70.3_{1.8}$ & $58.1_{2.4}$ & $59.5_{3.3}$ & $54.2_{3.8}$ & $3.3_{0.1}$ \\
CatBoost & $70.3_{1.6}$ & $36.1_{2.2}$ & $71.7_{2.7}$ & $40.7_{3.9}$ & $8.9_{0.3}$ \\
LightGBM & $65.7_{1.6}$ & $29.2_{2.1}$ & $68.3_{2.7}$ & $33.8_{3.7}$ & $4.3_{0.1}$ \\
Nadaraya-Watson & $52.4_{2.1}$ & $36.6_{2.4}$ & $67.9_{2.9}$ & $3